# AEFS: Adaptive Explainable Feature Selection for Cross-Project Software Defect Prediction

Self-contained pipeline notebook -- assembled from the tested `src/*.py` modules (see the accompanying `.py` files / README for the same code as importable modules).

**Stages**
1. Data Acquisition & Preprocessing (`load_tera`, `load_aeeem`)
2. Adaptive Explainable Feature Selection
   - 2A RUP filter (`rup.py`)
   - 2B Sequential Forward Selection wrapper
   - 2C SHAP refinement
   - 2D Cross-project stability / transferability check
3. Cross-Project Model Training & Prediction (LightGBM, Random Forest)
4. Baselines (All-Features / Filter-MI / Wrapper-RFE / Ablation) + statistical significance testing

**Data leakage note:** normalization, RUP filtering, and SMOTE are all fit ONLY on each fold's source/training projects, never on the held-out target project. See inline comments throughout for the reasoning.

**Checkpointing note:** the full Stage 4 run (bottom of this notebook) saves results to CSV after EVERY fold, not just at the end. If your session gets interrupted (Kaggle time limits, local machine sleep, etc.), just re-run the same cell -- it will detect the partial CSV and resume from wherever it left off instead of starting over.

## 0. Setup

In [ ]:
# If running on Kaggle/Colab, uncomment:
# !pip install -q lightgbm shap imbalanced-learn
!pip install -q lightgbm shap imbalanced-learn tqdm matplotlib seaborn python-dotenv huggingface_hub

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='shap')

from pathlib import Path

# Progress bar: auto-detect notebook vs CLI
try:
    from IPython import get_ipython
    _ipy = get_ipython()
    IN_NOTEBOOK = _ipy is not None and hasattr(_ipy, 'kernel')
except Exception:
    IN_NOTEBOOK = False

if IN_NOTEBOOK:
    from tqdm.notebook import tqdm
else:
    from tqdm import tqdm

print('Environment:', 'Jupyter Notebook' if IN_NOTEBOOK else 'CLI')


### Data locations

Point these at wherever your raw CSVs live. On Kaggle, this is typically `/kaggle/input/<your-dataset-name>/...` after attaching a dataset with the raw `tera/` and `aeeem/` folders uploaded.

In [ ]:
# Kaggle: upload AEFS_data.zip as dataset 'aefs-data', then Add Data in notebook settings
RAW_TERA_DIR = Path('/kaggle/input/datasets/mihjihad/aefs-data/tera')
RAW_AEEEM_DIR = Path('/kaggle/input/datasets/mihjihad/aefs-data/aeeem')

assert RAW_TERA_DIR.exists(), f'{RAW_TERA_DIR} not found -- update RAW_TERA_DIR above'
assert RAW_AEEEM_DIR.exists(), f'{RAW_AEEEM_DIR} not found -- update RAW_AEEEM_DIR above'
print('TeraPromise files:', len(list(RAW_TERA_DIR.glob('*.csv'))))
print('AEEEM files:', len(list(RAW_AEEEM_DIR.glob('*.csv'))))


In [ ]:
# Where results/checkpoints get written -- Kaggle uses /kaggle/working/
Path('/kaggle/working/results').mkdir(parents=True, exist_ok=True)


## 1. Data Acquisition & Preprocessing

### 1a. TeraPromise / Jureczko loader

In [ ]:
"""
Loader for the TeraPromise / Jureczko CK-metric pool.

Each file in data/raw/tera/ is one project-version (e.g. ant-1.7.csv), with
columns: wmc, dit, noc, cbo, rfc, lcom, ca, ce, npm, lcom3, loc, dam, moa,
mfa, cam, ic, cbm, amc, max_cc, avg_cc, bug

'bug' is a defect COUNT (0, 1, 2, ...), not binary. We binarize it
(bug > 0 -> 1) since CPDP is framed here as a binary classification problem,
consistent with how MCC / AUC-ROC / F1 are defined in the workflow.
"""

import re

import pandas as pd

RAW_TERA_OUT = Path('/kaggle/working/results/tera')

TERA_TARGET_COL = "bug"
TERA_ID_LIKE_COLS = {"name", "name.1", "version"}  # dropped if present; not predictive metrics


def _project_name_from_filename(fname: str) -> str:
    """'ant-1.7.csv' -> 'ant-1.7' (keep version, since LOPO in this literature
    is typically done at the project-version level, e.g. ant-1.7 is a
    distinct 'project' from ant-1.6)."""
    return re.sub(r"\.csv$", "", fname)


def load_raw_tera() -> dict[str, pd.DataFrame]:
    """Load every TeraPromise CSV as-is, keyed by project-version name."""
    projects = {}
    for f in sorted(RAW_TERA_DIR.glob("*.csv")):
        df = pd.read_csv(f)
        projects[_project_name_from_filename(f.name)] = df
    return projects


def clean_tera_project(df: pd.DataFrame) -> pd.DataFrame:
    """Clean a single TeraPromise project dataframe:
    - drop non-metric identifier columns if present
    - binarize the bug count target
    - coerce metrics to numeric, drop rows that fail to parse
    - drop fully-duplicate rows
    - drop constant (zero-variance) feature columns
    """
    df = df.copy()

    # Drop non-metric identifier columns (some Jureczko files include a
    # project/version/name column that isn't a software metric)
    drop_cols = [c for c in df.columns if c.strip().lower() in TERA_ID_LIKE_COLS]
    df = df.drop(columns=drop_cols, errors="ignore")

    if TERA_TARGET_COL not in df.columns:
        raise ValueError(f"Expected target column '{TERA_TARGET_COL}' not found; got {list(df.columns)}")

    # Coerce all feature columns to numeric; anything that fails becomes NaN
    feature_cols = [c for c in df.columns if c != TERA_TARGET_COL]
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df[TERA_TARGET_COL] = pd.to_numeric(df[TERA_TARGET_COL], errors="coerce")

    # Drop rows with any missing values (metrics or target)
    df = df.dropna(axis=0, how="any")

    # Binarize target: any defect count > 0 becomes 1
    df[TERA_TARGET_COL] = (df[TERA_TARGET_COL] > 0).astype(int)
    df = df.rename(columns={TERA_TARGET_COL: "defective"})

    # Drop exact duplicate rows
    df = df.drop_duplicates()

    # NOTE: constant/low-variance feature pruning is intentionally NOT done
    # here. That is Stage 2A's job (RUP filter), applied consistently across
    # the whole pool (or per LOPO training fold) rather than per single
    # project file -- doing it here would desynchronize the column schema
    # across projects and break cross-project training.

    return df.reset_index(drop=True)


def load_and_clean_tera(min_rows: int = 20, min_defective: int = 2) -> dict[str, pd.DataFrame]:
    """Load + clean all TeraPromise projects, dropping any project-version
    that ends up too small or with too few defective examples to be usable
    as a LOPO source/target (needed for both training and evaluation)."""
    raw = load_raw_tera()
    cleaned = {}
    dropped = {}
    for name, df in raw.items():
        try:
            c = clean_tera_project(df)
        except ValueError as e:
            dropped[name] = str(e)
            continue
        n_pos = int(c["defective"].sum())
        if len(c) < min_rows or n_pos < min_defective or n_pos == len(c):
            dropped[name] = f"rows={len(c)}, defective={n_pos} (below thresholds or single-class)"
            continue
        cleaned[name] = c
    if dropped:
        print(f"[tera] Dropped {len(dropped)} project(s):")
        for k, v in dropped.items():
            print(f"    - {k}: {v}")

    _assert_consistent_schema(cleaned, pool_name="tera")
    return cleaned


def _assert_consistent_schema(cleaned: dict[str, pd.DataFrame], pool_name: str) -> None:
    """Guard against any project ending up with a different column set than
    the rest of the pool -- LOPO cross-project training requires a uniform
    schema across every project in a pool."""
    schemas = {name: tuple(df.columns) for name, df in cleaned.items()}
    unique_schemas = set(schemas.values())
    if len(unique_schemas) > 1:
        from collections import Counter
        counts = Counter(schemas.values())
        raise ValueError(
            f"[{pool_name}] Inconsistent column schema across projects: "
            f"{len(unique_schemas)} distinct schemas found (counts={dict(Counter(len(s) for s in schemas.values()))}). "
            f"All projects in a pool must share identical columns for LOPO training."
        )


def save_cleaned_tera(cleaned: dict[str, pd.DataFrame]) -> None:
    Path(".").mkdir(parents=True, exist_ok=True)
    for name, df in cleaned.items():
        df.to_csv(TERA_OUT_DIR / f"{name}.csv", index=False)

### 1b. AEEEM loader

In [ ]:
"""
Loader for the AEEEM pool (D'Ambros et al. bug prediction dataset).

Each file in data/raw/aeeem/ is one project (eq, jdt, lc, ml, pde), with a
combined schema of CK/OO metrics (ck_oo_*), coupling metrics (WCHU_*,
LDHH_*), CVS churn/entropy metrics (Cvs*Entropy), prior-bug-history counts,
an 'id' column, and a binary 'class' target ('buggy' / 'clean').

Each raw file has a couple of malformed leading rows (an exported header
comment dumped into row 1, a blank row 2) — these are dropped naturally by
the numeric coercion + dropna step, same as any other unparseable row.
"""

import re

import pandas as pd


AEEEM_OUT_DIR = Path('/kaggle/working/results/aeeem')

TARGET_COL = "class"
ID_LIKE_COLS = {"id"}

PROJECT_NAMES = {
    "eq": "equinox",
    "jdt": "jdt",
    "lc": "lucene",
    "ml": "mylyn",
    "pde": "pde",
}


def _sanitize_column_name(name: str) -> str:
    """LightGBM rejects feature names containing special JSON characters
    (e.g. the trailing ':' in 'numberOfBugsFoundUntil:'). Strip anything
    that isn't alphanumeric or underscore so column names are safe to use
    as model feature names everywhere downstream."""
    return re.sub(r"[^A-Za-z0-9_]", "", name)


def load_raw_aeeem() -> dict[str, pd.DataFrame]:
    projects = {}
    for f in sorted(RAW_AEEEM_DIR.glob("*.csv")):
        stem = f.stem.lower()
        name = PROJECT_NAMES.get(stem, stem)
        projects[name] = pd.read_csv(f)
    return projects


def clean_aeeem_project(df: pd.DataFrame) -> pd.DataFrame:
    """Clean a single AEEEM project dataframe:
    - drop id column
    - coerce metrics to numeric, drop rows that fail to parse (this also
      removes the known malformed export-artifact rows)
    - map class label buggy/clean -> 1/0
    - drop fully-duplicate rows
    - drop constant (zero-variance) feature columns
    """
    df = df.copy()
    df = df.drop(columns=[c for c in df.columns if c.strip().lower() in ID_LIKE_COLS], errors="ignore")
    df = df.rename(columns={c: _sanitize_column_name(c) for c in df.columns})

    if TARGET_COL not in df.columns:
        raise ValueError(f"Expected target column '{TARGET_COL}' not found; got {list(df.columns)}")

    feature_cols = [c for c in df.columns if c != TARGET_COL]
    for c in feature_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Normalize label strings before dropping NaNs so stray whitespace/case
    # doesn't silently get treated as missing
    df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.lower()
    df[TARGET_COL] = df[TARGET_COL].map({"buggy": 1, "clean": 0})

    df = df.dropna(axis=0, how="any")
    df[TARGET_COL] = df[TARGET_COL].astype(int)
    df = df.rename(columns={TARGET_COL: "defective"})

    df = df.drop_duplicates()

    # NOTE: constant/low-variance feature pruning is intentionally NOT done
    # here -- see matching note in load_tera.py. That is Stage 2A's job.

    return df.reset_index(drop=True)


def load_and_clean_aeeem(min_rows: int = 20, min_defective: int = 2) -> dict[str, pd.DataFrame]:
    raw = load_raw_aeeem()
    cleaned = {}
    dropped = {}
    for name, df in raw.items():
        try:
            c = clean_aeeem_project(df)
        except ValueError as e:
            dropped[name] = str(e)
            continue
        n_pos = int(c["defective"].sum())
        if len(c) < min_rows or n_pos < min_defective or n_pos == len(c):
            dropped[name] = f"rows={len(c)}, defective={n_pos} (below thresholds or single-class)"
            continue
        cleaned[name] = c
    if dropped:
        print(f"[aeeem] Dropped {len(dropped)} project(s):")
        for k, v in dropped.items():
            print(f"    - {k}: {v}")

    _assert_consistent_schema(cleaned, pool_name="aeeem")
    return cleaned


def _assert_consistent_schema(cleaned: dict[str, pd.DataFrame], pool_name: str) -> None:
    """Guard against any project ending up with a different column set than
    the rest of the pool -- LOPO cross-project training requires a uniform
    schema across every project in a pool."""
    schemas = {name: tuple(df.columns) for name, df in cleaned.items()}
    unique_schemas = set(schemas.values())
    if len(unique_schemas) > 1:
        from collections import Counter
        counts = Counter(schemas.values())
        raise ValueError(
            f"[{pool_name}] Inconsistent column schema across projects: "
            f"{len(unique_schemas)} distinct schemas found (counts={dict(Counter(len(s) for s in schemas.values()))}). "
            f"All projects in a pool must share identical columns for LOPO training."
        )


def save_cleaned_aeeem(cleaned: dict[str, pd.DataFrame]) -> None:
    Path(".").mkdir(parents=True, exist_ok=True)
    for name, df in cleaned.items():
        df.to_csv(AEEEM_OUT_DIR / f"{name}.csv", index=False)

### 1c. Load + clean both pools

Run this once -- both pools are kept in memory as `dict[project_name, DataFrame]` for everything downstream.

In [ ]:
tera_pool = load_and_clean_tera()
print(f'TeraPromise: {len(tera_pool)} usable project-versions')

aeeem_pool = load_and_clean_aeeem()
print(f'AEEEM: {len(aeeem_pool)} usable projects')


In [ ]:
# --- DIAGNOSTIC: check pool contents and fold viability ---
print('=== TeraPromise pool ===')
print(f'Total projects: {len(tera_pool)}')
if len(tera_pool) > 0:
    for name, df in list(tera_pool.items()):
        n_def = int(df['defective'].sum())
        print(f'  {name:20s}  rows={len(df):5d}  defective={n_def}')
else:
    print('  ** EMPTY POOL — all projects were dropped during cleaning **')
    # Show what happened during load
    raw = load_raw_tera()
    print(f'  Raw files found: {len(raw)}')
    for name, df in list(raw.items())[:5]:
        bug_col = 'bug' if 'bug' in df.columns else None
        print(f'    {name}: {len(df)} rows, cols={list(df.columns[:5])}... bug_col={bug_col}')

print()
print('=== AEEEM pool ===')
print(f'Total projects: {len(aeeem_pool)}')
for name, df in aeeem_pool.items():
    n_def = int(df['defective'].sum())
    print(f'  {name:20s}  rows={len(df):5d}  defective={n_def}')

print()
print('=== Fold viability (min_test_defective=1) ===')
for name, df in tera_pool.items():
    ok = df['defective'].sum() >= 1
    if not ok:
        print(f'  SKIP target {name}: 0 defective')
n_viable = sum(1 for df in tera_pool.values() if df['defective'].sum() >= 1)
print(f'Viable targets: {n_viable} / {len(tera_pool)}')

## Shared SMOTE helper

In [ ]:
"""Shared SMOTE application helper, used by lopo.py and baselines.py so both
apply oversampling identically (same k_neighbors capping logic, same
degenerate-case handling)."""

import pandas as pd
from imblearn.over_sampling import SMOTE


def apply_smote(X: pd.DataFrame, y: pd.Series, random_state: int = 42):
    """Fit + apply SMOTE on (X, y). Falls back to returning the data
    unchanged if there are too few minority-class samples to run SMOTE at
    all (degenerate small-fold edge case)."""
    minority_count = y.value_counts().min()
    if minority_count < 2:
        return X, y
    k_neighbors = min(5, minority_count - 1)
    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    return smote.fit_resample(X, y)

## Stage 2A: RUP (Rapid Uninformative Pruning) filter

In [ ]:
"""
Stage 2A: RUP (Rapid Uninformative Pruning) -- filter-based feature
selection, no ML model involved.

RUP removes three kinds of uninformative software metrics, in order:
  1. Low-variance features    -- barely vary across modules, carry little signal
  2. Irrelevant features      -- weak/no statistical association with the
                                  defect label (point-biserial correlation)
  3. Redundant features       -- pairs of features that are almost measuring
                                  the same thing (high pairwise correlation);
                                  the less relevant one of each pair is dropped

All three checks are computed on a single fold's TRAINING (source) data only
-- never on the held-out target project -- consistent with the per-fold
leakage-avoidance approach used for normalization/SMOTE in lopo.py. This
module is applied AFTER scaling and BEFORE SMOTE (see supervisor discussion:
filtering on real, unaltered data first gives cleaner variance/correlation
statistics than filtering after synthetic oversampling).
"""

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.stats import pointbiserialr
from sklearn.feature_selection import VarianceThreshold


@dataclass
class RupResult:
    selected_features: list[str]
    dropped_low_variance: list[str] = field(default_factory=list)
    dropped_irrelevant: list[str] = field(default_factory=list)
    dropped_redundant: list[str] = field(default_factory=list)
    n_candidates_in: int = 0
    n_candidates_out: int = 0


def _relevance_scores(X: pd.DataFrame, y: pd.Series) -> pd.Series:
    """Point-biserial correlation of each (continuous) feature with the
    binary defect label. Equivalent to Pearson correlation between the
    feature and a 0/1 target."""
    scores = {}
    for col in X.columns:
        try:
            r, _ = pointbiserialr(y, X[col])
        except Exception:
            r = 0.0
        scores[col] = abs(r) if not np.isnan(r) else 0.0
    return pd.Series(scores)


def compute_rup_features(
    X: pd.DataFrame,
    y: pd.Series,
    var_threshold: float = 1e-4,
    relevance_threshold: float = 0.02,
    redundancy_threshold: float = 0.90,
    min_features: int = 5,
) -> RupResult:
    """Apply the three RUP filters in sequence and return the surviving
    feature list plus a breakdown of what was dropped and why.

    NOTE on var_threshold: software metrics are typically right-skewed (a
    few large/complex modules stretch the range), so after Min-Max scaling
    most "normal" values compress near 0 -- this can make a genuinely
    informative feature look artificially low-variance. var_threshold is
    therefore kept as a near-zero safety net that only catches truly
    constant/near-constant columns; the relevance filter (point-biserial
    correlation with the defect label) is the primary driver of which
    features are dropped as uninformative, since it measures actual
    association with the target rather than raw numeric spread.

    min_features acts as a safety floor: if the filters would otherwise
    leave fewer than min_features columns (e.g. on a very small/sparse
    fold), the most relevant features are kept back in until the floor
    is met, so Stage 2B (the SHAP wrapper) always has a usable candidate
    set to search over.
    """
    n_in = X.shape[1]
    all_features = list(X.columns)

    # --- 1. Low-variance filter ---
    vt = VarianceThreshold(threshold=var_threshold)
    vt.fit(X)
    kept_mask = vt.get_support()
    kept_after_variance = [c for c, keep in zip(all_features, kept_mask) if keep]
    dropped_low_variance = [c for c in all_features if c not in kept_after_variance]

    working = X[kept_after_variance]

    # Relevance scores are computed once, on the post-variance-filter set,
    # and reused both to drop irrelevant features and to break ties during
    # redundancy pruning.
    relevance = _relevance_scores(working, y)

    # --- 2. Irrelevance filter ---
    kept_after_relevance = relevance[relevance >= relevance_threshold].index.tolist()
    dropped_irrelevant = [c for c in kept_after_variance if c not in kept_after_relevance]

    # --- 3. Redundancy filter ---
    # Among any pair of features correlated above redundancy_threshold,
    # drop the one with lower relevance to the target.
    working2 = working[kept_after_relevance]
    dropped_redundant: list[str] = []
    if len(kept_after_relevance) > 1:
        corr_matrix = working2.corr().abs()
        # Sort candidate features by relevance ascending, so when we scan
        # pairs we consistently drop the weaker one and keep the stronger.
        ordered = relevance.loc[kept_after_relevance].sort_values(ascending=False).index.tolist()
        survivors: list[str] = []
        removed: set[str] = set()
        for feat in ordered:
            if feat in removed:
                continue
            survivors.append(feat)
            for other in ordered:
                if other == feat or other in removed or other in survivors:
                    continue
                if corr_matrix.loc[feat, other] >= redundancy_threshold:
                    removed.add(other)
                    dropped_redundant.append(other)
        selected = survivors
    else:
        selected = kept_after_relevance

    # --- Safety floor ---
    if len(selected) < min_features:
        # Back-fill with the highest-relevance features not yet selected,
        # regardless of the variance/redundancy filters, so the wrapper
        # stage always has enough candidates to search over.
        remaining_ranked = relevance.drop(index=selected, errors="ignore").sort_values(ascending=False)
        needed = min_features - len(selected)
        backfill = remaining_ranked.head(needed).index.tolist()
        selected = selected + backfill
        # Reconcile the "dropped" lists so the diagnostics stay accurate
        dropped_low_variance = [c for c in dropped_low_variance if c not in backfill]
        dropped_irrelevant = [c for c in dropped_irrelevant if c not in backfill]
        dropped_redundant = [c for c in dropped_redundant if c not in backfill]

    return RupResult(
        selected_features=selected,
        dropped_low_variance=dropped_low_variance,
        dropped_irrelevant=dropped_irrelevant,
        dropped_redundant=dropped_redundant,
        n_candidates_in=n_in,
        n_candidates_out=len(selected),
    )

## LOPO fold generation (Stage 1 boundary -> Stage 2A)

For each fold: scale (train-only) -> RUP filter (Stage 2A, train-only) -> SMOTE (train-only). Also exposes the pre-RUP all-feature data and per-source-project labels needed later by Stage 2D and the Stage 4 baselines.

In [ ]:
"""
LOPO (Leave-One-Project-Out) fold generation and per-fold preprocessing.

This is where normalization and SMOTE actually get applied -- fit only on
each fold's source/training projects, never on the held-out target project,
to avoid leaking target-project information into training (see discussion
with supervisor: this is a data-leakage fix relative to the original
diagram, which grouped normalization/SMOTE under a single global Stage 1
step). Stage 1's cleaning (load_tera.py / load_aeeem.py) intentionally left
data unscaled and imbalanced for exactly this reason.
"""

from dataclasses import dataclass
from typing import Iterator

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler


TARGET_COL = "defective"


@dataclass
class LopoFold:
    target_name: str                 # held-out project (test set)
    source_names: list[str]          # remaining projects (training pool)
    X_train: pd.DataFrame            # normalized, RUP-filtered, SMOTE-balanced
    y_train: pd.Series
    X_test: pd.DataFrame             # normalized + filtered using train-fit stats only
    y_test: pd.Series
    n_train_before_smote: int
    n_train_after_smote: int
    selected_features: list[str]     # RUP's surviving candidate feature set
    rup_result: RupResult
    # Pre-SMOTE, per-source-project breakdown (real rows only, no synthetic
    # samples) -- needed by Stage 2D's stability/transferability check,
    # which evaluates feature importance project-by-project rather than on
    # the pooled/balanced training set.
    X_train_presmote: pd.DataFrame
    y_train_presmote: pd.Series
    train_project_labels: pd.Series
    # Pre-RUP, ALL original features, scaled, pre-SMOTE -- needed by Stage 4
    # baselines (Filter/Wrapper methods), which must start from the same
    # full feature space AEFS does, not from RUP's already-pruned output
    # (RUP is part of AEFS itself, so reusing its output would make the
    # baseline comparison unfair / not self-contained).
    X_train_all_features: pd.DataFrame
    X_test_all_features: pd.DataFrame


def generate_lopo_folds(
    pool: dict[str, pd.DataFrame],
    min_test_defective: int = 1,
    smote_random_state: int = 42,
    apply_rup: bool = True,
    rup_kwargs: dict | None = None,
) -> Iterator[LopoFold]:
    """Yield one LopoFold per project in the pool.

    For each fold, in order:
      1. One project is held out as the target (test set).
      2. All remaining projects are concatenated as the source (training pool).
      3. A MinMaxScaler is fit ONLY on the source data, then used to
         transform both source and target.
      4. RUP filtering (Stage 2A) is computed ONLY on the scaled source
         data, and the resulting candidate feature set is applied to both
         source and target.
      5. SMOTE is fit and applied ONLY on the (scaled, filtered) source data.
    """
    names = list(pool.keys())
    rup_kwargs = rup_kwargs or {}

    for target_name in names:
        test_df = pool[target_name]
        source_names = [n for n in names if n != target_name]
        source_dfs = [pool[n] for n in source_names]
        train_df = pd.concat(source_dfs, axis=0, ignore_index=True)
        # Parallel, row-aligned label of which source project each training
        # row came from -- built before SMOTE, since synthetic rows have no
        # single real project of origin.
        project_labels = pd.Series(
            np.concatenate([[n] * len(df) for n, df in zip(source_names, source_dfs)]),
            index=train_df.index,
        )

        # Skip folds where the held-out project has too few defective
        # examples to meaningfully evaluate MCC/AUC/F1 on
        if test_df[TARGET_COL].sum() < min_test_defective:
            continue

        feature_cols = [c for c in train_df.columns if c != TARGET_COL]

        X_train_raw = train_df[feature_cols]
        y_train_raw = train_df[TARGET_COL]
        X_test_raw = test_df[feature_cols]
        y_test = test_df[TARGET_COL]

        # 1. Normalize -- fit on training/source data only
        scaler = MinMaxScaler()
        X_train_scaled = pd.DataFrame(
            scaler.fit_transform(X_train_raw), columns=feature_cols, index=X_train_raw.index
        )
        X_test_scaled = pd.DataFrame(
            scaler.transform(X_test_raw), columns=feature_cols, index=X_test_raw.index
        )

        # 2. RUP filter (Stage 2A) -- fit on training/source data only
        if apply_rup:
            rup_result = compute_rup_features(X_train_scaled, y_train_raw, **rup_kwargs)
            selected = rup_result.selected_features
        else:
            rup_result = RupResult(selected_features=feature_cols, n_candidates_in=len(feature_cols),
                                    n_candidates_out=len(feature_cols))
            selected = feature_cols

        # Snapshot the full, pre-RUP feature space for Stage 4 baselines
        # (see LopoFold docstring above)
        X_train_all_features = X_train_scaled.copy()
        X_test_all_features = X_test_scaled.copy()

        X_train_scaled = X_train_scaled[selected]
        X_test_scaled = X_test_scaled[selected]

        # Snapshot pre-SMOTE, per-project-labeled data for Stage 2D
        X_train_presmote = X_train_scaled.copy()
        y_train_presmote = y_train_raw.copy()

        # 3. SMOTE -- fit/applied on training/source data only, after
        #    scaling AND filtering
        n_before = len(X_train_scaled)
        X_train_final, y_train_final = apply_smote(X_train_scaled, y_train_raw, random_state=smote_random_state)
        n_after = len(X_train_final)

        yield LopoFold(
            target_name=target_name,
            source_names=source_names,
            X_train=X_train_final,
            y_train=y_train_final,
            X_test=X_test_scaled,
            y_test=y_test,
            n_train_before_smote=n_before,
            n_train_after_smote=n_after,
            selected_features=selected,
            rup_result=rup_result,
            X_train_presmote=X_train_presmote,
            y_train_presmote=y_train_presmote,
            train_project_labels=project_labels,
            X_train_all_features=X_train_all_features,
            X_test_all_features=X_test_all_features,
        )

## Stage 2B (SFS wrapper) + 2C (SHAP refinement) + 2D (stability check)

In [ ]:
"""
Stage 2B/2C/2D: the "wrapper + explainability" half of AEFS, applied after
RUP (Stage 2A) has already produced a candidate feature set.

  2B - Sequential Forward Selection (SFS) wrapper: starting from an empty
       set, repeatedly add whichever remaining RUP candidate feature most
       improves cross-validated MCC using a LightGBM base learner, on the
       pooled (SMOTE-balanced) training data. Stops when no candidate
       improves the score by at least `min_improvement`.

  2C - SHAP refinement: train LightGBM on 2B's best subset, compute SHAP
       values (TreeExplainer), rank features by mean |SHAP value|, and drop
       any feature contributing less than `min_contribution_frac` of the
       total SHAP mass.

  2D - Stability / transferability check: using 2C's refined feature set,
       go back to each individual SOURCE project's own (pre-SMOTE, real-row
       only) data, train a small model per project, rank that project's
       top-K most important features by SHAP, and keep only features that
       appear in the top-K set for at least `stability_threshold` fraction
       of source projects. This is the step that actually tests
       transferability rather than just in-pool accuracy.

All of this operates within a single LOPO fold and never touches the
held-out target project's data.
"""

from dataclasses import dataclass, field
from itertools import combinations
import math

import numpy as np
import pandas as pd
import shap
from lightgbm import LGBMClassifier
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import StratifiedKFold

# Keep LightGBM quiet and fast -- these are wrapper-search models, not the
# final Stage 3 production model, so a modest tree count is fine.
LGBM_PARAMS = dict(
    n_estimators=60,
    max_depth=5,
    num_leaves=15,
    verbosity=-1,
    n_jobs=-1,
)


@dataclass
class Sfs2BResult:
    selected_features: list[str]
    score_history: list[tuple[str, float]] = field(default_factory=list)  # (feature added, cv MCC after adding)


@dataclass
class Shap2CResult:
    selected_features: list[str]
    dropped_low_contribution: list[str] = field(default_factory=list)
    mean_abs_shap: dict = field(default_factory=dict)  # feature -> mean |SHAP|


@dataclass
class Stability2DResult:
    selected_features: list[str]
    dropped_unstable: list[str] = field(default_factory=list)
    presence_rate: dict = field(default_factory=dict)   # feature -> fraction of source projects with it in top-K
    n_projects_evaluated: int = 0
    n_projects_skipped: int = 0
    mean_pairwise_jaccard: float = 0.0


def _cv_mcc(X: pd.DataFrame, y: pd.Series, k: int = 3, random_state: int = 42) -> float:
    """Mean MCC of a LightGBM classifier under k-fold stratified CV."""
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)
    scores = []
    for train_idx, val_idx in skf.split(X, y):
        clf = LGBMClassifier(**LGBM_PARAMS, random_state=random_state)
        clf.fit(X.iloc[train_idx], y.iloc[train_idx])
        preds = clf.predict(X.iloc[val_idx])
        scores.append(matthews_corrcoef(y.iloc[val_idx], preds))
    return float(np.mean(scores))


def sequential_forward_selection(
    X: pd.DataFrame,
    y: pd.Series,
    candidates: list[str],
    cv_folds: int = 3,
    min_improvement: float = 0.001,
    max_features: int | None = None,
    min_features: int = 5,
    random_state: int = 42,
) -> Sfs2BResult:
    """Stage 2B: greedy forward search over `candidates`, scored by
    cross-validated MCC. Stops early once adding the best remaining
    candidate no longer improves the score by at least min_improvement.
    Always keeps at least min_features to avoid starving downstream steps."""
    remaining = list(candidates)
    selected: list[str] = []
    best_score = -1.0
    history: list[tuple[str, float]] = []
    max_features = max_features or len(candidates)

    pbar = tqdm(total=min(len(candidates), max_features), desc='  [2B] SFS', leave=False)
    while remaining and len(selected) < max_features:
        scores = {}
        for feat in remaining:
            trial = selected + [feat]
            scores[feat] = _cv_mcc(X[trial], y, k=cv_folds, random_state=random_state)

        best_feat = max(scores, key=scores.get)
        best_trial_score = scores[best_feat]

        if best_trial_score - best_score < min_improvement and selected:
            # No meaningful gain from adding another feature -- stop.
            break

        selected.append(best_feat)
        remaining.remove(best_feat)
        best_score = best_trial_score
        history.append((best_feat, best_score))
        pbar.update(1)
        pbar.set_postfix(features=len(selected), mcc=f'{best_score:.4f}')
    pbar.close()

    if len(selected) < min_features and remaining:
        fallback_pool = list(remaining)
        scores_fb = {}
        for feat in fallback_pool:
            trial = selected + [feat]
            scores_fb[feat] = _cv_mcc(X[trial], y, k=cv_folds, random_state=random_state)
        while len(selected) < min_features and fallback_pool:
            best_fb = max(scores_fb, key=scores_fb.get)
            selected.append(best_fb)
            fallback_pool.remove(best_fb)
            scores_fb.pop(best_fb)
            for feat in list(scores_fb):
                trial = selected + [feat]
                scores_fb[feat] = _cv_mcc(X[trial], y, k=cv_folds, random_state=random_state)

    return Sfs2BResult(selected_features=selected, score_history=history)


def shap_refine(
    X: pd.DataFrame,
    y: pd.Series,
    candidates: list[str],
    min_contribution_frac: float = 0.02,
    min_features: int = 5,
    random_state: int = 42,
) -> Shap2CResult:
    """Stage 2C: train on `candidates`, rank by mean |SHAP|, drop features
    contributing less than min_contribution_frac of total SHAP mass."""
    clf = LGBMClassifier(**LGBM_PARAMS, random_state=random_state)
    clf.fit(X[candidates], y)

    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X[candidates])
    # LightGBM binary classification via shap can return either a single
    # (n_samples, n_features) array or a list of two such arrays (one per
    # class) depending on version -- handle both.
    if isinstance(shap_values, list):
        shap_values = shap_values[1]  # positive (defective) class
    if shap_values.ndim == 3:
        # (n_samples, n_features, n_classes) layout in some shap versions
        shap_values = shap_values[:, :, 1]

    mean_abs = np.abs(shap_values).mean(axis=0)
    importance = pd.Series(mean_abs, index=candidates).sort_values(ascending=False)
    total = importance.sum()
    contribution_frac = importance / total if total > 0 else importance

    keep = contribution_frac[contribution_frac >= min_contribution_frac].index.tolist()
    if len(keep) < min_features:
        keep = importance.head(min_features).index.tolist()

    dropped = [f for f in candidates if f not in keep]

    return Shap2CResult(
        selected_features=keep,
        dropped_low_contribution=dropped,
        mean_abs_shap=importance.to_dict(),
    )


def _jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)


def stability_transferability_check(
    X_presmote: pd.DataFrame,
    y_presmote: pd.Series,
    project_labels: pd.Series,
    candidates: list[str],
    top_k: int | None = None,
    top_k_fraction: float = 0.7,
    stability_threshold: float = 0.3,
    min_project_rows: int = 20,
    min_project_minority: int = 2,
    min_features: int = 5,
    random_state: int = 42,
) -> Stability2DResult:
    """Stage 2D: per-source-project SHAP importance ranking + Jaccard
    stability check over `candidates` (Stage 2C's refined feature set).

    A feature is retained only if it appears in a source project's top-K
    important features for at least `stability_threshold` fraction of the
    source projects that could be evaluated (projects too small or with
    too few minority-class rows are skipped, not counted against features).

    top_k defaults to ceil(top_k_fraction * len(candidates)) rather than a
    flat 50% cut. Rationale: with a small number of source projects (e.g.
    4 in AEEEM), requiring BOTH "in the top half of importance" AND
    "present in >= stability_threshold of only a handful of projects" is a
    compounding double-strictness that can discard features which are
    consistently moderately useful everywhere (never top-half in any single
    project, but reliably contributing across all of them). Widening top_k
    to ~70% relaxes the first filter so the second (cross-project presence)
    is the one doing the real discriminating work. This is a mechanism-level
    adjustment, not something tuned against held-out target performance.
    """
    if top_k is None:
        top_k = max(1, math.ceil(top_k_fraction * len(candidates)))

    project_top_sets: dict[str, set] = {}
    skipped = 0

    projects = list(project_labels.unique())
    proj_pbar = tqdm(projects, desc='  [2D] Stability', leave=False)
    for proj in proj_pbar:
        proj_pbar.set_postfix(proj=proj)
        mask = project_labels == proj
        Xp = X_presmote.loc[mask, candidates]
        yp = y_presmote.loc[mask]

        if len(Xp) < min_project_rows or yp.value_counts().min() < min_project_minority:
            skipped += 1
            continue

        clf = LGBMClassifier(**LGBM_PARAMS, random_state=random_state)
        clf.fit(Xp, yp)
        explainer = shap.TreeExplainer(clf)
        shap_values = explainer.shap_values(Xp)
        if isinstance(shap_values, list):
            shap_values = shap_values[1]
        if shap_values.ndim == 3:
            shap_values = shap_values[:, :, 1]

        mean_abs = np.abs(shap_values).mean(axis=0)
        ranked = pd.Series(mean_abs, index=candidates).sort_values(ascending=False)
        project_top_sets[proj] = set(ranked.head(top_k).index)
    proj_pbar.close()

    n_evaluated = len(project_top_sets)

    if n_evaluated == 0:
        # Degenerate case (e.g. every source project too small) -- fall
        # back to keeping everything rather than emptying the feature set.
        return Stability2DResult(
            selected_features=candidates,
            n_projects_evaluated=0,
            n_projects_skipped=skipped,
        )

    presence_rate = {
        feat: sum(feat in s for s in project_top_sets.values()) / n_evaluated
        for feat in candidates
    }

    selected = [f for f in candidates if presence_rate[f] >= stability_threshold]
    if len(selected) < min_features:
        ranked_by_presence = sorted(candidates, key=lambda f: presence_rate[f], reverse=True)
        selected = ranked_by_presence[:min_features]

    dropped = [f for f in candidates if f not in selected]

    # Mean pairwise Jaccard similarity across source projects' top-K sets --
    # a single summary number for how consistent feature importance is
    # across the source pool this fold (useful to report as a
    # "transferability" statistic later in Stage 4).
    proj_names = list(project_top_sets.keys())
    if len(proj_names) >= 2:
        pairs = combinations(proj_names, 2)
        jaccards = [_jaccard(project_top_sets[a], project_top_sets[b]) for a, b in pairs]
        mean_jaccard = float(np.mean(jaccards))
    else:
        mean_jaccard = 1.0

    return Stability2DResult(
        selected_features=selected,
        dropped_unstable=dropped,
        presence_rate=presence_rate,
        n_projects_evaluated=n_evaluated,
        n_projects_skipped=skipped,
        mean_pairwise_jaccard=mean_jaccard,
    )

### Quick sanity check: run 2A->2D on a single fold
Swap `aeeem_pool` for `tera_pool` to check the other pool (TeraPromise will be slower).

In [ ]:
_fold = next(iter(generate_lopo_folds(aeeem_pool)))
print(f'Target: {_fold.target_name}')
print(f'RUP candidates (2A): {len(_fold.selected_features)}')

_sfs = sequential_forward_selection(_fold.X_train, _fold.y_train, _fold.selected_features)
print(f'[2B] SFS selected: {_sfs.selected_features}')

_shap = shap_refine(_fold.X_train, _fold.y_train, _sfs.selected_features)
print(f'[2C] SHAP refined: {_shap.selected_features}')

_stab = stability_transferability_check(
    _fold.X_train_presmote, _fold.y_train_presmote, _fold.train_project_labels,
    _shap.selected_features,
)
print(f'[2D] Final adaptive feature set: {_stab.selected_features}')
print(f'[2D] Mean pairwise Jaccard: {_stab.mean_pairwise_jaccard:.3f}')


## Stage 3: Cross-Project Model Training and Prediction

In [ ]:
"""
Stage 3: Cross-Project Model Training and Prediction.

For each LOPO fold, using the FINAL adaptive feature set produced by Stage
2D, train both ensemble classifiers named in the workflow (LightGBM and
Random Forest) on the source/training data, then predict on the held-out
target project.

Hyperparameters here are deliberately more robust than the lightweight ones
used inside Stage 2B's internal SFS search (which needed to run hundreds of
times per fold and prioritized speed over polish). These are the
"production" models whose predictions actually get scored in Stage 4, so a
fixed, reasonable, IDENTICAL configuration is used across every fold and
every project, for both models -- this keeps the comparison fair (no
per-fold hyperparameter tuning that could advantage some folds over others)
and keeps runtime predictable. If you want hyperparameter tuning later,
that would need to be its own explicit, cross-validated step -- flagging
this as an assumption rather than deciding it silently.
"""

from dataclasses import dataclass

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42

LGBM_FINAL_PARAMS = dict(
    n_estimators=300,
    max_depth=-1,
    num_leaves=31,
    learning_rate=0.05,
    verbosity=-1,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)

RF_FINAL_PARAMS = dict(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)


@dataclass
class FoldPrediction:
    target_name: str
    model_name: str              # "lightgbm" or "random_forest"
    features_used: list[str]
    y_true: pd.Series
    y_pred: np.ndarray           # predicted labels (0/1) at 0.5 threshold
    y_proba: np.ndarray          # predicted probability of the positive (defective) class


def train_and_predict(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    features: list[str],
    target_name: str,
) -> list[FoldPrediction]:
    """Train both ensemble classifiers on `features` only, predict on the
    held-out target project, and return one FoldPrediction per model."""
    results = []

    Xtr = X_train[features]
    Xte = X_test[features]

    lgbm = LGBMClassifier(**LGBM_FINAL_PARAMS)
    lgbm.fit(Xtr, y_train)
    lgbm_proba = lgbm.predict_proba(Xte)[:, 1]
    results.append(FoldPrediction(
        target_name=target_name, model_name="lightgbm", features_used=features,
        y_true=y_test, y_pred=(lgbm_proba >= 0.5).astype(int), y_proba=lgbm_proba,
    ))

    rf = RandomForestClassifier(**RF_FINAL_PARAMS)
    rf.fit(Xtr, y_train)
    rf_proba = rf.predict_proba(Xte)[:, 1]
    results.append(FoldPrediction(
        target_name=target_name, model_name="random_forest", features_used=features,
        y_true=y_test, y_pred=(rf_proba >= 0.5).astype(int), y_proba=rf_proba,
    ))

    return results

## Stage 4: Baseline feature selection methods

All baselines except the Ablation start from the SAME pre-RUP, all-feature, scaled data AEFS itself starts from -- RUP is part of AEFS's own pipeline, so reusing its output would make the baseline comparison unfair.

In [ ]:
"""
Stage 4: Baseline feature selection methods AEFS is compared against.

All baselines (except the ablation) start from the SAME pre-RUP, all-feature,
scaled data that AEFS itself starts from (fold.X_train_all_features) -- not
from RUP's output, since RUP is part of AEFS's own pipeline and reusing it
would make the "baseline" not self-contained. Each baseline selects its own
K features (K = AEFS's final feature count for that fold, for fairness),
then SMOTE is (re-)applied fresh on just those K columns before training.

  - All Features:     no selection at all (the floor)
  - Filter (MI):       SelectKBest with mutual_info_classif
  - Wrapper (RFE):      Recursive Feature Elimination with a LightGBM estimator
  - Ablation:          AEFS's own Stage 2C output (SHAP-refined), WITHOUT
                        the Stage 2D stability/transferability check --
                        isolates whether 2D specifically adds value
"""

from dataclasses import dataclass

import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.feature_selection import SelectKBest, mutual_info_classif, RFE


RANDOM_STATE = 42

# Lightweight LightGBM config for RFE's internal ranking model -- this is a
# feature-selection tool, not the final production model (mirrors the
# speed-vs-polish tradeoff already used in Stage 2B's SFS search).
RFE_ESTIMATOR_PARAMS = dict(n_estimators=60, max_depth=5, num_leaves=15, verbosity=-1, n_jobs=-1)


def select_filter_mi(X: pd.DataFrame, y: pd.Series, k: int, random_state: int = RANDOM_STATE) -> list[str]:
    k = min(k, X.shape[1])
    selector = SelectKBest(score_func=lambda X_, y_: mutual_info_classif(X_, y_, random_state=random_state), k=k)
    selector.fit(X, y)
    return list(X.columns[selector.get_support()])


def select_wrapper_rfe(X: pd.DataFrame, y: pd.Series, k: int, random_state: int = RANDOM_STATE) -> list[str]:
    k = min(k, X.shape[1])
    estimator = LGBMClassifier(**RFE_ESTIMATOR_PARAMS, random_state=random_state)
    selector = RFE(estimator=estimator, n_features_to_select=k, step=0.2)
    selector.fit(X, y)
    return list(X.columns[selector.get_support()])


def run_baseline(
    fold,
    selector_fn,
    k: int,
    label: str,
    smote_random_state: int = RANDOM_STATE,
) -> list[FoldPrediction]:
    """Select features via `selector_fn` on the fold's full pre-RUP feature
    space, re-apply SMOTE on just those columns, train + predict."""
    X_pre = fold.X_train_all_features
    y_pre = fold.y_train_presmote  # same rows/order as X_train_all_features (RUP only drops columns, not rows)

    selected = selector_fn(X_pre, y_pre, k)

    X_train_final, y_train_final = apply_smote(X_pre[selected], y_pre, random_state=smote_random_state)
    X_test_final = fold.X_test_all_features[selected]

    return train_and_predict(X_train_final, y_train_final, X_test_final, fold.y_test,
                              selected, fold.target_name)


def run_all_features_baseline(fold) -> list[FoldPrediction]:
    """No selection -- train on every original feature. SMOTE was already
    applied to the full feature space in lopo.py's apply_rup=True pass IF
    RUP happened to keep everything, but in general we need our own SMOTE
    run on the untouched full feature set (fold.X_train_all_features is
    pre-SMOTE), so this is not simply fold.X_train."""
    X_train_final, y_train_final = apply_smote(fold.X_train_all_features, fold.y_train_presmote,
                                                random_state=RANDOM_STATE)
    return train_and_predict(X_train_final, y_train_final, fold.X_test_all_features, fold.y_test,
                              list(fold.X_train_all_features.columns), fold.target_name)

## Statistical significance testing (Wilcoxon, Friedman + Nemenyi post-hoc)

In [ ]:
"""
Stage 4: statistical significance testing.

- Wilcoxon signed-rank test: paired comparison of AEFS vs one baseline
  across folds (same target projects), per metric.
- Friedman test: omnibus test for whether >=3 methods differ significantly
  across folds, per metric.
- Nemenyi post-hoc: pairwise comparison after a significant Friedman result,
  implemented manually (critical-difference formula, Demsar 2006) since
  scikit-posthocs has a dependency conflict with the installed statsmodels
  version in this environment.
"""

import itertools

import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, friedmanchisquare, rankdata

# Studentized range statistic q_alpha (two-tailed, alpha=0.05, infinite df),
# indexed by number of methods k. Standard Nemenyi test reference values
# (Demsar, 2006, "Statistical Comparisons of Classifiers over Multiple Data
# Sets", Table). CD = q_alpha * sqrt(k(k+1) / (6N)).
NEMENYI_Q_ALPHA_005 = {
    2: 1.960, 3: 2.343, 4: 2.569, 5: 2.728, 6: 2.850,
    7: 2.949, 8: 3.031, 9: 3.102, 10: 3.164,
}


def wilcoxon_vs_baseline(scores_a: pd.Series, scores_b: pd.Series) -> dict:
    """Paired Wilcoxon signed-rank test: scores_a (e.g. AEFS) vs scores_b
    (a baseline), matched by index (fold/target project). Returns the
    statistic, p-value, and whether a is significantly better on average."""
    diffs = scores_a.values - scores_b.values
    if np.allclose(diffs, 0):
        return {"statistic": 0.0, "p_value": 1.0, "mean_diff": 0.0, "n": len(diffs), "note": "identical scores"}
    try:
        stat, p = wilcoxon(scores_a, scores_b)
    except ValueError as e:
        return {"statistic": None, "p_value": None, "mean_diff": float(np.mean(diffs)),
                "n": len(diffs), "note": str(e)}
    return {
        "statistic": float(stat),
        "p_value": float(p),
        "mean_diff": float(np.mean(diffs)),
        "n": len(diffs),
        "significant_at_0.05": bool(p < 0.05),
    }


def friedman_test(scores_matrix: pd.DataFrame) -> dict:
    """scores_matrix: rows = folds/targets, columns = methods. Omnibus test
    for whether the methods' score distributions differ significantly."""
    columns = [scores_matrix[c].values for c in scores_matrix.columns]
    stat, p = friedmanchisquare(*columns)
    return {"statistic": float(stat), "p_value": float(p), "significant_at_0.05": bool(p < 0.05),
            "n_methods": scores_matrix.shape[1], "n_folds": scores_matrix.shape[0]}


def nemenyi_posthoc(scores_matrix: pd.DataFrame, alpha: float = 0.05) -> pd.DataFrame:
    """Manual Nemenyi post-hoc test following a significant Friedman result.
    Returns a DataFrame of pairwise method comparisons with the rank
    difference, critical difference (CD), and whether it's significant.

    Methods are ranked per fold (1 = best), averaged across folds, and any
    pair whose average-rank difference exceeds CD is considered
    significantly different at the given alpha.
    """
    k = scores_matrix.shape[1]
    n = scores_matrix.shape[0]
    if k not in NEMENYI_Q_ALPHA_005:
        raise ValueError(f"No tabulated q_alpha for k={k} methods; add it to NEMENYI_Q_ALPHA_005 "
                          f"(supported: {sorted(NEMENYI_Q_ALPHA_005)})")
    q_alpha = NEMENYI_Q_ALPHA_005[k]
    cd = q_alpha * np.sqrt(k * (k + 1) / (6 * n))

    # Rank methods per fold; higher score = better = rank 1. rankdata ranks
    # ascending, so rank the negated scores.
    ranks = scores_matrix.apply(lambda row: rankdata(-row.values), axis=1, result_type="expand")
    ranks.columns = scores_matrix.columns
    avg_ranks = ranks.mean(axis=0)

    rows = []
    for a, b in itertools.combinations(scores_matrix.columns, 2):
        diff = abs(avg_ranks[a] - avg_ranks[b])
        rows.append({
            "method_a": a, "method_b": b,
            "avg_rank_a": round(avg_ranks[a], 3), "avg_rank_b": round(avg_ranks[b], 3),
            "rank_diff": round(diff, 3), "critical_difference": round(cd, 3),
            "significant": bool(diff > cd),
        })
    return pd.DataFrame(rows)

## Full orchestration: AEFS + all baselines, across every LOPO fold

This is the full Stage 2A->4 pipeline, run per fold, for both models, for AEFS and all 4 baselines. **Runtime:** ~5-6 minutes for AEEEM (5 folds), **~2.5 hours for TeraPromise (68 folds)** -- see the note below before running TeraPromise.

**Checkpointed & resumable**: results are saved to CSV after every fold. If this cell is interrupted partway through, just re-run it -- it will pick up where it left off.

In [ ]:
"""
Stage 4: full comparison of AEFS against its 4 baselines across every LOPO
fold in a pool, for both ensemble models, followed by statistical
significance testing (Wilcoxon AEFS-vs-each-baseline, Friedman + Nemenyi
across all methods), run separately per model since AEFS's benefit may not
be identical for LightGBM vs Random Forest.
"""

import time
import warnings

import pandas as pd
from sklearn.metrics import matthews_corrcoef, f1_score, roc_auc_score


warnings.filterwarnings("ignore", category=UserWarning, module="shap")

METHODS = ["AEFS", "Ablation_no2D", "AllFeatures", "FilterMI", "WrapperRFE"]


def _score_row(pool_name, target, method, mean_jaccard, n_features, p):
    mcc = matthews_corrcoef(p.y_true, p.y_pred)
    f1 = f1_score(p.y_true, p.y_pred, zero_division=0)
    try:
        auc = roc_auc_score(p.y_true, p.y_proba)
    except ValueError:
        auc = float("nan")
    return {
        "pool": pool_name, "target": target, "method": method, "model": p.model_name,
        "n_features": n_features, "mean_jaccard": mean_jaccard,
        "test_mcc": round(mcc, 4), "test_f1": round(f1, 4), "test_auc_roc": round(auc, 4),
    }


def run_stage4_pool(
    pool: dict[str, pd.DataFrame],
    pool_name: str,
    checkpoint_path: str | Path | None = None,
) -> pd.DataFrame:
    """Run AEFS + all baselines across every LOPO fold in `pool`.

    If `checkpoint_path` is given, results are (1) loaded from that CSV at
    startup if it already exists, so any folds already completed are
    SKIPPED rather than redone, and (2) re-saved to that same path after
    EVERY fold completes, not just at the end. This makes long runs
    (TeraPromise's 68 folds can take ~2.5 hours) resumable after an
    interruption -- restart the script and it picks up where it left off
    instead of starting over from fold 1.
    """
    rows: list[dict] = []
    completed_targets: set[str] = set()

    if checkpoint_path is not None:
        checkpoint_path = Path(checkpoint_path)
        if checkpoint_path.exists():
            existing = pd.read_csv(checkpoint_path)
            existing = existing[existing["pool"] == pool_name]
            rows = existing.to_dict("records")
            completed_targets = set(existing["target"].unique())
            if completed_targets:
                print(f"[{pool_name}] Resuming from checkpoint: {len(completed_targets)} "
                      f"fold(s) already done ({checkpoint_path})")

    folds = list(generate_lopo_folds(pool))
    remaining = [f for f in folds if f.target_name not in completed_targets]
    print(f"[{pool_name}] {len(folds)} LOPO folds total, {len(remaining)} remaining "
          f"x {len(METHODS)} methods x 2 models\n")

    fold_pbar = tqdm(remaining, desc=f'[{pool_name}] Folds', position=0, leave=True)
    total_t0 = time.time()
    for i, fold in enumerate(fold_pbar, 1):
        t0 = time.time()
        fold_pbar.set_description(f'[{pool_name}] Fold {i}/{len(remaining)} ({fold.target_name})')

        # --- AEFS's own pipeline (2A already done inside the fold; 2B/2C/2D here) ---
        sfs_result = sequential_forward_selection(fold.X_train, fold.y_train, fold.selected_features)
        shap_result = shap_refine(fold.X_train, fold.y_train, sfs_result.selected_features)
        stability_result = stability_transferability_check(
            fold.X_train_presmote, fold.y_train_presmote, fold.train_project_labels,
            shap_result.selected_features,
        )
        k = len(stability_result.selected_features)
        jac = round(stability_result.mean_pairwise_jaccard, 3)

        aefs_preds = train_and_predict(fold.X_train, fold.y_train, fold.X_test, fold.y_test,
                                        stability_result.selected_features, fold.target_name)
        for p in aefs_preds:
            rows.append(_score_row(pool_name, fold.target_name, "AEFS", jac, k, p))

        # --- Ablation: Stage 2C output, no Stage 2D stability check ---
        ablation_preds = train_and_predict(fold.X_train, fold.y_train, fold.X_test, fold.y_test,
                                            shap_result.selected_features, fold.target_name)
        for p in ablation_preds:
            rows.append(_score_row(pool_name, fold.target_name, "Ablation_no2D", None,
                                    len(shap_result.selected_features), p))

        # --- All Features baseline ---
        all_preds = run_all_features_baseline(fold)
        for p in all_preds:
            rows.append(_score_row(pool_name, fold.target_name, "AllFeatures", None,
                                    len(fold.X_train_all_features.columns), p))

        # --- Filter (MI) baseline, matched to k ---
        filter_preds = run_baseline(fold, select_filter_mi, k, "FilterMI")
        for p in filter_preds:
            rows.append(_score_row(pool_name, fold.target_name, "FilterMI", None, k, p))

        # --- Wrapper (RFE) baseline, matched to k ---
        wrapper_preds = run_baseline(fold, select_wrapper_rfe, k, "WrapperRFE")
        for p in wrapper_preds:
            rows.append(_score_row(pool_name, fold.target_name, "WrapperRFE", None, k, p))

        elapsed = time.time() - t0
        total_elapsed = time.time() - total_t0
        eta = (total_elapsed / i * (len(remaining) - i)) if i > 0 else 0
        eta_str = f'{eta/60:.1f}min' if eta >= 60 else f'{eta:.0f}s'
        fold_pbar.set_postfix(
            target=fold.target_name, k=k,
            elapsed=f'{elapsed:.0f}s',
            eta=eta_str,
        )
        print(f"  [{i}/{len(remaining)}] target={fold.target_name:12s} AEFS_k={k}  ({elapsed:.1f}s)  ETA: {eta_str}")

        # Checkpoint after every fold, not just at the end
        if checkpoint_path is not None:
            pd.DataFrame(rows).to_csv(checkpoint_path, index=False)

    fold_pbar.close()
    total_time = time.time() - total_t0
    print(f"\n[{pool_name}] Completed {len(remaining)} folds in {total_time/60:.1f} minutes")
    return pd.DataFrame(rows)


def run_significance_tests(df: pd.DataFrame, metric: str = "test_mcc"):
    """Run Wilcoxon (AEFS vs each baseline) and Friedman+Nemenyi (all
    methods), separately per model."""
    for model in df["model"].unique():
        print(f"\n{'='*70}\nModel: {model}  |  Metric: {metric}\n{'='*70}")
        sub = df[df["model"] == model]
        pivot = sub.pivot(index="target", columns="method", values=metric)[METHODS]
        print(pivot.round(4))

        print(f"\n-- Wilcoxon signed-rank: AEFS vs each baseline ({metric}) --")
        for baseline in METHODS[1:]:
            result = wilcoxon_vs_baseline(pivot["AEFS"], pivot[baseline])
            sig = result.get("significant_at_0.05", "n/a")
            print(f"  AEFS vs {baseline:15s} mean_diff={result['mean_diff']:+.4f}  "
                  f"p={result['p_value']}  significant={sig}")

        print(f"\n-- Friedman omnibus test (all {len(METHODS)} methods, {metric}) --")
        fried = friedman_test(pivot)
        print(f"  statistic={fried['statistic']:.3f}  p={fried['p_value']:.4f}  "
              f"significant={fried['significant_at_0.05']}  (n_folds={fried['n_folds']})")

        if fried["n_folds"] >= 3:
            print(f"\n-- Nemenyi post-hoc ({metric}) --")
            nemenyi = nemenyi_posthoc(pivot)
            print(nemenyi.to_string(index=False))

### Run on AEEEM (fast -- recommended first, to confirm everything works end to end)

In [ ]:
import time

_aeeem_ckpt = '/kaggle/working/results/aeeem_stage4_full_results.csv'
t0 = time.time()
aeeem_results = run_stage4_pool(aeeem_pool, 'aeeem', checkpoint_path=_aeeem_ckpt)
print(f'Total time: {time.time()-t0:.0f}s')

aeeem_results.groupby(['method', 'model'])[['test_mcc', 'test_f1', 'test_auc_roc']].mean().round(4)


In [ ]:
for metric in ['test_mcc', 'test_f1', 'test_auc_roc']:
    run_significance_tests(aeeem_results, metric=metric)


### Run on TeraPromise (SLOW -- ~2.5 hours)

Recommend running this as a background/detached process rather than a blocking notebook cell if your platform kills long-idle kernels. On Kaggle, a plain blocking cell is usually fine as long as you keep the session active; if it does get interrupted, just re-run the cell -- checkpointing means it resumes automatically rather than starting over from fold 1.

**A quick single-fold timing check is recommended first** -- run the cell below before committing to the full 68-fold loop.

In [ ]:
# Quick single-fold timing check before running the full pool
_t0 = time.time()
_fold = next(iter(generate_lopo_folds(tera_pool)))
_sfs = sequential_forward_selection(_fold.X_train, _fold.y_train, _fold.selected_features)
_shap = shap_refine(_fold.X_train, _fold.y_train, _sfs.selected_features)
_stab = stability_transferability_check(_fold.X_train_presmote, _fold.y_train_presmote,
                                         _fold.train_project_labels, _shap.selected_features)
_ = train_and_predict(_fold.X_train, _fold.y_train, _fold.X_test, _fold.y_test,
                      _stab.selected_features, _fold.target_name)
print(f'One fold (partial, AEFS only): {time.time()-_t0:.0f}s -- '
      f'full fold w/ baselines is ~2x this; x68 folds for a rough total estimate')


In [ ]:
_tera_ckpt = '/kaggle/working/results/tera_stage4_full_results.csv'
t0 = time.time()
tera_results = run_stage4_pool(tera_pool, 'tera', checkpoint_path=_tera_ckpt)
print(f'Total time this session: {(time.time()-t0)/60:.1f} minutes')

tera_results.groupby(['method', 'model'])[['test_mcc', 'test_f1', 'test_auc_roc']].mean().round(4)


In [ ]:
for metric in ['test_mcc', 'test_f1', 'test_auc_roc']:
    run_significance_tests(tera_results, metric=metric)


## Stage 5: Generate Publication Figures
All figures are saved as both PDF (vector, for the paper) and PNG (for presentations/preview).

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=1.1)
PALETTE = {'AEFS': '#e63946', 'Ablation_no2D': '#457b9d',
           'AllFeatures': '#2a9d8f', 'FilterMI': '#e9c46a', 'WrapperRFE': '#264653'}
FIG_DIR = '/kaggle/working/figures'
os.makedirs(FIG_DIR, exist_ok=True)

def save_fig(name, fig):
    fig.savefig(os.path.join(FIG_DIR, f'{name}.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(FIG_DIR, f'{name}.png'), dpi=200, bbox_inches='tight')
    plt.close(fig)
    print(f'  Saved: {name}.pdf / .png')

In [ ]:
# --- Figure 1: Grouped bar chart — mean MCC per method per model ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    summary = pool_df.groupby(['method', 'model'])['test_mcc'].mean().reset_index()
    g = sns.catplot(data=summary, x='method', y='test_mcc', hue='model',
                    kind='bar', palette='Set2', height=5, aspect=1.6, legend_out=True)
    g.set_axis_labels('Method', 'Mean MCC')
    g.set_xticklabels(rotation=25, ha='right')
    g.ax.set_title(f'{pool_name} — Mean MCC by Method and Model')
    for container in g.ax.containers:
        g.ax.bar_label(container, fmt='%.3f', fontsize=8)
    save_fig(f'{pool_name}_fig1_mcc_grouped_bar', g.fig)


In [ ]:
# --- Figure 2: Box plot — MCC distribution across folds ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    fig, ax = plt.subplots(figsize=(10, 5))
    order = ['AEFS', 'Ablation_no2D', 'AllFeatures', 'FilterMI', 'WrapperRFE']
    sns.boxplot(data=pool_df, x='method', y='test_mcc', order=order,
                palette=PALETTE, ax=ax)
    sns.stripplot(data=pool_df, x='method', y='test_mcc', order=order,
                  color='black', size=3, alpha=0.4, ax=ax)
    ax.set_title(f'{pool_name} — MCC Distribution Across LOPO Folds')
    ax.set_xlabel('')
    ax.set_ylabel('MCC')
    ax.tick_params(axis='x', rotation=20)
    save_fig(f'{pool_name}_fig2_mcc_boxplot', fig)


In [ ]:
# --- Figure 3: Feature count comparison ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    feat_summary = pool_df.groupby(['method', 'model'])['n_features'].mean().reset_index()
    g = sns.catplot(data=feat_summary, x='method', y='n_features', hue='model',
                    kind='bar', palette='Set2', height=5, aspect=1.6)
    g.set_axis_labels('Method', 'Mean # Features Selected')
    g.set_xticklabels(rotation=25, ha='right')
    g.ax.set_title(f'{pool_name} — Average Feature Count by Method')
    for container in g.ax.containers:
        g.ax.bar_label(container, fmt='%.1f', fontsize=8)
    save_fig(f'{pool_name}_fig3_feature_count', g.fig)


In [ ]:
# --- Figure 4: Ablation study — AEFS vs Ablation_no2D vs AllFeatures (lightgbm only) ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    sub = pool_df[pool_df['model'] == 'lightgbm']
    ablation_methods = ['AEFS', 'Ablation_no2D', 'AllFeatures']
    sub2 = sub[sub['method'].isin(ablation_methods)]
    summary = sub2.groupby('method')['test_mcc'].agg(['mean', 'std']).reindex(ablation_methods)
    fig, ax = plt.subplots(figsize=(7, 5))
    colors = [PALETTE[m] for m in ablation_methods]
    bars = ax.bar(summary.index, summary['mean'], yerr=summary['std'],
                  color=colors, capsize=5, edgecolor='black', linewidth=0.5)
    ax.set_ylabel('MCC')
    ax.set_title(f'{pool_name} — Ablation Study (LightGBM)')
    for bar, val in zip(bars, summary['mean']):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=10)
    save_fig(f'{pool_name}_fig4_ablation', fig)


In [ ]:
# --- Figure 5: Per-project MCC heatmap (LightGBM) ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    sub_all = pool_df[pool_df['model'] == 'lightgbm']
    pivot = sub_all.pivot_table(index='target', columns='method', values='test_mcc')
    fig, ax = plt.subplots(figsize=(10, max(6, len(pivot) * 0.35)))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
                linewidths=0.5, ax=ax, cbar_kws={'label': 'MCC'})
    ax.set_title(f'{pool_name} — Per-Project MCC Heatmap (LightGBM)')
    ax.set_ylabel('Target Project')
    ax.set_xlabel('Method')
    save_fig(f'{pool_name}_fig5_heatmap', fig)


In [ ]:
# --- Figure 6: Jaccard stability distribution (AEFS folds only) ---
for pool_name, pool_df in [('AEEEM', aeeem_results), ('TeraPromise', tera_results)]:
    jac_data = pool_df[pool_df['method'] == 'AEFS'][['target', 'mean_jaccard']].dropna()
    if jac_data.empty:
        print(f'  [{pool_name}] No Jaccard data — skipping')
        continue
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.histplot(jac_data['mean_jaccard'], bins=15, kde=True, color=PALETTE['AEFS'], ax=ax)
    ax.axvline(jac_data['mean_jaccard'].mean(), color='red', linestyle='--',
               label=f'Mean = {jac_data["mean_jaccard"].mean():.3f}')
    ax.set_title(f'{pool_name} — Pairwise Jaccard Stability (AEFS)')
    ax.set_xlabel('Mean Jaccard Similarity')
    ax.legend()
    save_fig(f'{pool_name}_fig6_jaccard', fig)


In [ ]:
# List all generated figures
print('\nAll generated figures:')
for f in sorted(os.listdir(FIG_DIR)):
    size_kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
    print(f'  {f:50s} {size_kb:7.1f} KB')

## Upload Results & Figures to Hugging Face
Set your HF token in the `.env` file as `HF_TOKEN=hf_...`.  
The upload target repo is set below — change `HF_REPO` if needed.

In [ ]:
from huggingface_hub import HfApi, login

# On Kaggle use Secrets; locally fall back to .env
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('Loaded HF_TOKEN from Kaggle Secrets')
except Exception:
    from dotenv import load_dotenv
    load_dotenv()
    HF_TOKEN = os.getenv('HF_TOKEN')
    if HF_TOKEN:
        print('Loaded HF_TOKEN from .env file')

if not HF_TOKEN:
    raise RuntimeError(
        'HF_TOKEN not found. On Kaggle: add it via Add Data > Add Environment Variable. '
        'Locally: put HF_TOKEN=hf_... in your .env file.')

login(token=HF_TOKEN)
api = HfApi()

HF_REPO = 'MoshinAli/aefs-cpdp-results'

# Create repo if it doesn't exist (private by default)
try:
    api.create_repo(HF_REPO, repo_type='dataset', private=True, exist_ok=True)
    print(f'Repo ready: https://huggingface.co/datasets/{HF_REPO}')
except Exception as e:
    print(f'Repo setup: {e}')


In [ ]:
# Upload all results CSVs + figures
upload_files = []

# Result CSVs
results_dir = '/kaggle/working/results'
for f in os.listdir(results_dir):
    if f.endswith('.csv'):
        upload_files.append(os.path.join(results_dir, f))

# Figures
for f in os.listdir(FIG_DIR):
    upload_files.append(os.path.join(FIG_DIR, f))

# Also upload the notebook itself
upload_files.append('/kaggle/working/AEFS_Pipeline.ipynb')

print(f'Uploading {len(upload_files)} files to {HF_REPO}:\n')
for fp in upload_files:
    fname = os.path.basename(fp)
    # Organize into subdirectories on HF
    if fp.endswith('.csv'):
        hf_path = f'results/{fname}'
    elif fp.endswith(('.pdf', '.png')):
        hf_path = f'figures/{fname}'
    else:
        hf_path = fname
    try:
        api.upload_file(
            path_or_fileobj=fp,
            path_in_repo=hf_path,
            repo_id=HF_REPO,
            repo_type='dataset',
        )
        print(f'  OK: {hf_path}')
    except Exception as e:
        print(f'  FAIL: {hf_path} — {e}')

print(f'\nDone. Browse: https://huggingface.co/datasets/{HF_REPO}/tree/main')
